In [ ]:
import os, math, json, joblib, warnings, gc
import numpy as np, pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import lightgbm as lgb

In [ ]:
warnings.filterwarnings("ignore")
SEED = 42
MODEL_DIR = "/content/models"
os.makedirs(MODEL_DIR, exist_ok=True)

def time_split_per_user_indices(df_input, user_col="user_id", test_frac=0.2):
    train_idx, test_idx = [], []
    users = df_input[user_col].unique()
    for u in users:
        idxs = df_input[df_input[user_col]==u].index.tolist()
        if len(idxs)==0: continue
        n_test = max(1, int(math.ceil(len(idxs)*test_frac)))
        train_idx.extend(idxs[:-n_test])
        test_idx.extend(idxs[-n_test:])
    return sorted(train_idx), sorted(test_idx)

print("Setup done. MODEL_DIR:", MODEL_DIR)


Setup done. MODEL_DIR: /content/models


In [ ]:
# Cell 2
import os, pandas as pd
PATH = "/content/merged_features.csv"
if not os.path.exists(PATH):
    raise FileNotFoundError(f"{PATH} not found — upload merged_features.csv to /content and re-run this cell.")
df = pd.read_csv(PATH, parse_dates=["date","intervention_start","intervention_end","week_start"], low_memory=False)
print("Loaded merged_features.csv shape:", df.shape)
display(df.head(2))


Loaded merged_features.csv shape: (731000, 90)


,user_id,date,week_start,workday,profession,work_mode,chronotype,age,sex,height_cm,...,mood_volatility_7d,stress_trend,under_intervention,mood_change_during_intervention,stress_reduction_post_intervention,sleep_deficit_norm,stress_level_norm,workload_index_norm,screen_stress_ratio_norm,health_twin_index
0,1,2024-01-01,2024-01-01,1,operations,onsite,morning,27,female,174,...,0.0,0.0,0,0.0,0.0,0.540995,0.375,0.343946,0.259933,48.895147
1,110,2024-01-01,2024-01-01,1,teacher,onsite,morning,40,female,165,...,0.0,0.0,0,0.0,0.0,0.610286,0.500,0.473316,0.191430,53.691420


In [ ]:
# Cell 3
df = df.sort_values(["user_id","date"]).reset_index(drop=True)

# Next-day targets (safe overwrite)
df["mood_next_day"] = df.groupby("user_id")["mood_score"].shift(-1)
df["stress_next_day"] = df.groupby("user_id")["stress_level"].shift(-1)
df["energy_next_day"] = df.groupby("user_id")["energy_level"].shift(-1)

# Keep only rows with all three next-day targets
df_td = df.dropna(subset=["mood_next_day","stress_next_day","energy_next_day"]).reset_index(drop=True)
print("df_td rows (with next-day targets):", len(df_td))


df_td rows (with next-day targets): 730000


In [ ]:
# Choose features and save feature list
# Cell 4
import numpy as np
numeric_cols = df_td.select_dtypes(include=[np.number]).columns.tolist()
exclude = {"mood_next_day","stress_next_day","energy_next_day","user_id"}
feature_cols = [c for c in numeric_cols if c not in exclude]

# If you have a tailored feature list you want to use, replace feature_cols above.
print("Feature count:", len(feature_cols))
print("Example features:", feature_cols[:30])

# Save
feat_file = os.path.join(MODEL_DIR,"feature_cols_multioutput.csv")
pd.Series(feature_cols).to_csv(feat_file, index=False, header=False)
print("Saved feature list to:", feat_file)


Feature count: 75
Example features: ['workday', 'age', 'height_cm', 'baseline_bmi', 'sleep_hours', 'sleep_quality', 'work_hours', 'meetings_count', 'tasks_completed', 'emails_received', 'commute_minutes', 'exercise_minutes', 'steps_count', 'caffeine_mg', 'alcohol_units', 'screen_time_hours', 'social_interactions', 'outdoor_time_minutes', 'diet_quality', 'calories_intake', 'stress_level', 'mood_score', 'energy_level', 'focus_score', 'weather_mood_impact', 'weight_kg', 'job_satisfaction_x', 'perceived_stress_scale_x', 'anxiety_score_x', 'depression_score_x']
Saved feature list to: /content/models/feature_cols_multioutput.csv


In [ ]:
# Fit & save preprocessing pipeline (median impute + StandardScaler)
# Cell 5
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

preproc = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
X_all_raw = df_td[feature_cols]
preproc.fit(X_all_raw)
joblib.dump(preproc, os.path.join(MODEL_DIR,"preproc_multioutput.joblib"))
print("Saved preproc to:", os.path.join(MODEL_DIR,"preproc_multioutput.joblib"))


Saved preproc to: /content/models/preproc_multioutput.joblib


In [ ]:
# Time-aware split and transform (train/test)
# Cell 6
train_idx, test_idx = time_split_per_user_indices(df_td, user_col="user_id", test_frac=0.2)

X_train_raw = df_td.loc[train_idx, feature_cols]
X_test_raw  = df_td.loc[test_idx, feature_cols]
Y_train = df_td.loc[train_idx, ["mood_next_day","stress_next_day","energy_next_day"]]
Y_test  = df_td.loc[test_idx,  ["mood_next_day","stress_next_day","energy_next_day"]]

preproc = joblib.load(os.path.join(MODEL_DIR,"preproc_multioutput.joblib"))
Xtr_p = preproc.transform(X_train_raw)
Xte_p = preproc.transform(X_test_raw)

Xtr_p_df = pd.DataFrame(Xtr_p, columns=feature_cols, index=X_train_raw.index)
Xte_p_df = pd.DataFrame(Xte_p, columns=feature_cols, index=X_test_raw.index)

print("Train/test shapes:", Xtr_p_df.shape, Xte_p_df.shape, Y_train.shape, Y_test.shape)


Train/test shapes: (584000, 75) (146000, 75) (584000, 3) (146000, 3)


In [ ]:
# RandomForest baseline (multi-output) and save
# Cell 7
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor

rf_base = RandomForestRegressor(n_estimators=120, max_depth=12, min_samples_leaf=4, random_state=SEED, n_jobs=1)
multi_rf = MultiOutputRegressor(rf_base, n_jobs=1)
print("Training RandomForest baseline ...")
multi_rf.fit(Xtr_p_df, Y_train)

Ypred_rf = multi_rf.predict(Xte_p_df)
for i, col in enumerate(Y_train.columns):
    print(f"RF baseline - {col}: MSE={mean_squared_error(Y_test.iloc[:,i], Ypred_rf[:,i]):.4f}, R2={r2_score(Y_test.iloc[:,i], Ypred_rf[:,i]):.4f}")

joblib.dump(multi_rf, os.path.join(MODEL_DIR,"global_multioutput_rf.joblib"))
print("Saved RF baseline to", os.path.join(MODEL_DIR,"global_multioutput_rf.joblib"))


Training RandomForest baseline ...
RF baseline - mood_next_day: MSE=1.6539, R2=0.2244
RF baseline - stress_next_day: MSE=0.9245, R2=0.3314
RF baseline - energy_next_day: MSE=2.1700, R2=0.0382
Saved RF baseline to /content/models/global_multioutput_rf.joblib


In [ ]:
# Global LightGBM (default) and save
# Cell 8
from sklearn.multioutput import MultiOutputRegressor
import lightgbm as lgb

lgb_params = {"n_estimators":300, "learning_rate":0.05, "num_leaves":31, "random_state":SEED}
base = lgb.LGBMRegressor(**lgb_params)
global_lgb = MultiOutputRegressor(base, n_jobs=1)
print("Training global LightGBM (default) ...")
global_lgb.fit(Xtr_p_df, Y_train)

Ypred_lgb = global_lgb.predict(Xte_p_df)
for i, col in enumerate(Y_train.columns):
    print(f"Global LGB - {col}: MSE={mean_squared_error(Y_test.iloc[:,i], Ypred_lgb[:,i]):.4f}, R2={r2_score(Y_test.iloc[:,i], Ypred_lgb[:,i]):.4f}")

joblib.dump(global_lgb, os.path.join(MODEL_DIR,"global_multioutput_lgb_default.joblib"))
print("Saved default global LGB to", os.path.join(MODEL_DIR,"global_multioutput_lgb_default.joblib"))


Training global LightGBM (default) ...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.543988 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7966
[LightGBM] [Info] Number of data points in the train set: 584000, number of used features: 75
[LightGBM] [Info] Start training from score 6.164716
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.171241 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7966
[LightGBM] [Info] Number of data points in the train set: 584000, number of used features: 75
[LightGBM] [Info] Start training from score 4.217875
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.175289 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_c